In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, DoubleType, DateType, TimestampType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime

print("Libraries imported successfully.")

In [0]:
spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.databricks.delta.properties.defaults.enableChangeDataFeed", "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "8")   # keep low for dev cluster

print(f"✅ Spark version  : {spark.version}")
print(f"✅ App name       : {spark.conf.get('spark.app.name', 'Unknown')}")

In [0]:
CATALOG        = "databricks_project_tsv-752"
SCHEMA         = "healthcare"
BASE_VOL       = f"/Volumes/{CATALOG}/{SCHEMA}"

RAW_PATH           = f"{BASE_VOL}/raw/"
BRONZE_PATH        = f"{BASE_VOL}/bronze/"
SILVER_PATH        = f"{BASE_VOL}/silver/"
GOLD_PATH          = f"{BASE_VOL}/gold/"
CHECKPOINT_PATH    = f"{BASE_VOL}/checkpoints/"
STREAMING_LAND     = f"{BASE_VOL}/streaming/"

VISITS_CSV         = f"{RAW_PATH}patient_visits.csv"
VISITS_B2_CSV      = f"{RAW_PATH}patient_visits_batch2.csv"
PATIENTS_CSV       = f"{RAW_PATH}patients.csv"
BILLING_CSV        = f"{RAW_PATH}billing.csv"

In [0]:
required_files = [VISITS_CSV, PATIENTS_CSV, BILLING_CSV, VISITS_B2_CSV]

for f in required_files:
    try:
        info = dbutils.fs.ls(f)
        print(f"✅ Found: {f}  ({info[0].size:,} bytes)")
    except Exception:
        print(f"❌ MISSING: {f}  — upload this file via Data > Add Data > DBFS")

In [0]:
# Catalog and schema already exist — just set the context
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
print(f"✅ Context set to: {CATALOG}.{SCHEMA}")

In [0]:
df_visits_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(VISITS_CSV)
)

print("=" * 60)
print("📄 Schema: patient_visits.csv")
print("=" * 60)
df_visits_raw.printSchema()
print(f"Row count: {df_visits_raw.count():,}")
display(df_visits_raw.limit(5))

In [0]:
df_patients_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATIENTS_CSV)
)

print("=" * 60)
print("📄 Schema: patients.csv")
print("=" * 60)
df_patients_raw.printSchema()
print(f"Row count: {df_patients_raw.count():,}")
display(df_patients_raw.limit(5))

In [0]:
df_billing_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(BILLING_CSV)
)

print("=" * 60)
print("📄 Schema: billing.csv")
print("=" * 60)
df_billing_raw.printSchema()
print(f"Row count: {df_billing_raw.count():,}")
display(df_billing_raw.limit(5))

In [0]:
def add_bronze_metadata(df, source_file_hint=None):
    """
    Adds _ingested_at (current UTC timestamp) and _source_file columns.
    _metadata.file_path returns the actual file path when reading from files.
    If the DataFrame was built in memory (tests), we fall back to the hint string.
    """
    df = df.withColumn("_ingested_at", F.current_timestamp())

    try:
        df = df.withColumn("_source_file", F.col("_metadata.file_path"))
    except Exception:
        df = df.withColumn("_source_file", F.lit(source_file_hint or "unknown"))

    return df

In [0]:
df_visits_bronze   = add_bronze_metadata(df_visits_raw)
df_patients_bronze = add_bronze_metadata(df_patients_raw)
df_billing_bronze  = add_bronze_metadata(df_billing_raw)

print("✅ Metadata columns added to all three DataFrames.")
display(df_visits_bronze.select("visit_id", "_ingested_at", "_source_file").limit(3))

In [0]:
BRONZE_VISITS_TABLE   = f"`{CATALOG}`.`{SCHEMA}`.bronze_patient_visits"
BRONZE_PATIENTS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.bronze_patients"
BRONZE_BILLING_TABLE  = f"`{CATALOG}`.`{SCHEMA}`.bronze_billing"
BRONZE_VISITS_STREAM_TABLE = f"`{CATALOG}`.`{SCHEMA}`.bronze_patient_visits_stream"

In [0]:
try:
    spark.sql(f"""
        ALTER TABLE {BRONZE_VISITS_TABLE}
        ALTER COLUMN visit_id SET NOT NULL
    """)
    print(f"✅ NOT NULL constraint set on visit_id  in {BRONZE_VISITS_TABLE}")
except Exception as e:
    print(f"ℹ️  NOT NULL may already be set: {e}")

# Confirm — printSchema should show visit_id as nullable=false
spark.table(BRONZE_VISITS_TABLE).printSchema()

In [0]:
print("🧪 Testing CHECK constraint — inserting negative consultation_fee...")

try:
    bad_row = spark.createDataFrame(
        [("BILL_BAD", "V999", -500, 100.0, 200.0, 50.0, "pending")],
        ["bill_id", "visit_id", "consultation_fee",
         "medication_cost", "procedure_cost", "insurance_covered", "payment_status"]
    ).withColumn("_ingested_at", F.current_timestamp()) \
     .withColumn("_source_file", F.lit("test"))

    bad_row.write.format("delta").mode("append").saveAsTable(BRONZE_BILLING_TABLE)
    print("❌ UNEXPECTED: Bad row was inserted — constraint not working!")
except Exception as e:
    print(f"✅ EXPECTED FAILURE — constraint blocked bad row.\n   Error: {e}")

In [0]:
(
    df_visits_bronze        # same DataFrame as before
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_VISITS_TABLE)
)

print(f"✅ Second overwrite of {BRONZE_VISITS_TABLE} complete → version 1 created.")

In [0]:
print("=" * 70)
print(f"📜 DESCRIBE HISTORY: {BRONZE_VISITS_TABLE}")
print("=" * 70)

history_df = spark.sql(f"DESCRIBE HISTORY {BRONZE_VISITS_TABLE}")
display(history_df.select("version", "timestamp", "operation", "operationMetrics"))

# Assertion: must have at least 2 versions
version_count = history_df.count()
assert version_count >= 2, (
    f"Expected ≥ 2 versions in history, found {version_count}. "
    f"Re-run Cell 12 and Cell 18 in sequence."
)
print(f"\n✅ Verified: {version_count} version(s) exist in transaction log.")

In [0]:
df_v0 = spark.sql(f"SELECT * FROM {BRONZE_VISITS_TABLE} VERSION AS OF 0")

print(f"Version 0 row count : {df_v0.count():,}")
print(f"Current row count   : {spark.table(BRONZE_VISITS_TABLE).count():,}")
display(df_v0.limit(3))

In [0]:
BRONZE_VISITS_STREAM_TABLE = f"`{CATALOG}`.`{SCHEMA}`.bronze_patient_visits_stream"
AUTOLOADER_CHECKPOINT      = CHECKPOINT_PATH + "bronze_visits_stream/"

print("🚀 Starting Auto Loader stream for batch-2 visits...")

# ── Schema hint so Auto Loader does not need to scan files for schema ──
visits_schema_hint = (
    "visit_id STRING, patient_id STRING, doctor_id STRING, "
    "visit_date STRING, department STRING, visit_type STRING, "
    "duration_days INT, discharge_status STRING"
)

stream_query = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "schema_visits_stream/")
    .option("header", "true")
    .schema(visits_schema_hint)          # use hint for deterministic schema
    .load(RAW_PATH)                      # monitors the entire raw folder for new CSVs
    # ─── Add metadata columns on the streaming DataFrame ───────────────
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))   # Databricks Runtime ≥ 11.3
    # ─── Write to Bronze stream table ──────────────────────────────────
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", AUTOLOADER_CHECKPOINT)
    .trigger(availableNow=True)          # process all available files then stop
    .toTable(BRONZE_VISITS_STREAM_TABLE)
)

# Wait for the micro-batch to complete
stream_query.awaitTermination(timeout=120)

print(f"✅ Auto Loader ingestion complete → {BRONZE_VISITS_STREAM_TABLE}")
print(f"   Rows loaded: {spark.table(BRONZE_VISITS_STREAM_TABLE).count():,}")

In [0]:
print("=" * 70)
print(f"📜 DESCRIBE HISTORY: {BRONZE_VISITS_STREAM_TABLE}")
print("=" * 70)
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_VISITS_STREAM_TABLE}")
        .select("version", "timestamp", "operation", "operationMetrics"))

print("\n📊 Sample rows from stream table:")
display(spark.table(BRONZE_VISITS_STREAM_TABLE).limit(5))

In [0]:

(
    df_patients_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PATIENTS_TABLE)
)
print(f"✅ {BRONZE_PATIENTS_TABLE} written — {spark.table(BRONZE_PATIENTS_TABLE).count():,} rows")

(
    df_billing_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_BILLING_TABLE)
)
print(f"✅ {BRONZE_BILLING_TABLE} written — {spark.table(BRONZE_BILLING_TABLE).count():,} rows")

In [0]:
print("=" * 70)
print("🏆 BRONZE LAYER — INGESTION SUMMARY")
print("=" * 70)

bronze_tables = {
    "patient_visits"        : BRONZE_VISITS_TABLE,
    "patients"              : BRONZE_PATIENTS_TABLE,
    "billing"               : BRONZE_BILLING_TABLE,
    "patient_visits_stream" : BRONZE_VISITS_STREAM_TABLE,
}

for label, table in bronze_tables.items():
    try:
        cnt = spark.table(table).count()
        print(f"  ✅ {label:30s}  {cnt:>6,} rows   →  {table}")
    except Exception as e:
        print(f"  ❌ {label:30s}  ERROR: {e}")

print("\n📋 Constraints on bronze.billing:")
spark.sql(f"SHOW TBLPROPERTIES {BRONZE_BILLING_TABLE}").show(truncate=False)

print("\n📋 Schema of bronze.patient_visits (check visit_id nullable=false):")
spark.table(BRONZE_VISITS_TABLE).printSchema()